[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fuellabs-ai/cookbooks/blob/main/what-changed/walkthrough.ipynb)

# What really changed in v2?

A walkthrough for the `what_changed.py` recipe. The idea: word diffs show every
edit, but most edits are noise. An embedder can say which edits changed the
meaning and which just changed the words.

Model: `harims95/hobbylm-embed-140m` through Sentence Transformers. Embeddings
only, no LLM. Clauses are embedded as plain text, no query/passage prompts,
because this is a similarity job, not a search job. The model already returns
unit-length vectors, so similarity is a plain dot product.

Sample data: every file in `examples/` is fiction, written for this recipe.


In [1]:
%pip install -q sentence-transformers==6.3.0

# Repo path for downloads: keep in sync with the Colab badge link above.
REPO = "fuellabs-ai/cookbooks"
BRANCH = "main"
RAW_BASE = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/what-changed"

# Offline fallback: point WHAT_CHANGED_SRC at a local checkout of this recipe
# folder and the files are copied from there when the raw URLs are unreachable
# (for example before the repo goes public).
import os, shutil, urllib.request

LOCAL_SRC = os.environ.get("WHAT_CHANGED_SRC", "")
ROLES = ["gtm_pricing_page", "ecommerce_supplier_spec", "sales_proposal_v7", "npd_product_brief"]
FILES = ["what_changed.py"]
for role in ROLES:
    FILES += [f"examples/{role}/{name}" for name in ("old.txt", "new.txt", "hard_cases.csv")]

def fetch(rel):
    if os.path.exists(rel):
        return "already present"
    try:
        os.makedirs(os.path.dirname(rel) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_BASE}/{rel}", rel)
        return "downloaded from raw.githubusercontent.com"
    except Exception:
        pass
    if LOCAL_SRC:
        local = os.path.join(LOCAL_SRC, rel)
        if os.path.exists(local):
            shutil.copy(local, rel)
            return f"copied from local checkout ({LOCAL_SRC})"
    raise FileNotFoundError(
        f"could not fetch {rel}; the repo may not be public yet. "
        "Set WHAT_CHANGED_SRC to a local checkout of the recipe folder and run again."
    )

for rel in FILES:
    print(f"{fetch(rel)}: {rel}")

Note: you may need to restart the kernel to use updated packages.
already present: what_changed.py
already present: examples/gtm_pricing_page/old.txt
already present: examples/gtm_pricing_page/new.txt
already present: examples/gtm_pricing_page/hard_cases.csv
already present: examples/ecommerce_supplier_spec/old.txt
already present: examples/ecommerce_supplier_spec/new.txt
already present: examples/ecommerce_supplier_spec/hard_cases.csv
already present: examples/sales_proposal_v7/old.txt
already present: examples/sales_proposal_v7/new.txt
already present: examples/sales_proposal_v7/hard_cases.csv
already present: examples/npd_product_brief/old.txt
already present: examples/npd_product_brief/new.txt
already present: examples/npd_product_brief/hard_cases.csv


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("harims95/hobbylm-embed-140m", revision="ea708cee107818fa0c1871eebd26d82a1e3fb1aa", trust_remote_code=True)
print("embedding dimension:", model.get_embedding_dimension())

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 120/120 [00:00<00:00, 9685.12it/s]

embedding dimension: 512


## Step 1: split both versions into clauses

Long paragraphs match badly. The script splits on lines, bullet markers and
sentence boundaries, so every clause is short and comparable. The model reads
512 tokens by default, up to 2,048 if you raise the limit, and short clauses
keep the matching sharp.

In [3]:
from what_changed import split_clauses

old_text = open("examples/gtm_pricing_page/old.txt").read()
new_text = open("examples/gtm_pricing_page/new.txt").read()

old_clauses = split_clauses(old_text)
new_clauses = split_clauses(new_text)
print(len(old_clauses), "old clauses,", len(new_clauses), "new clauses")
for c in old_clauses[:6]:
    print(" -", c)

22 old clauses, 27 new clauses
 - [Sample data: fictional pricing page for Nimbus Metrics, a made-up analytics tool]
 - Nimbus Metrics pricing
 - Starter plan
 - $29 per month.
 - Track up to 10,000 events per month.
 - Includes 3 dashboards and email support.


## Step 2: embed every clause, match by dot product

Vectors come out normalized (length 1), so the dot product is the cosine
similarity. Each new clause looks for its closest old clause.

In [4]:
import numpy as np

probe_pairs = [
    ("$79 per month.", "$89 per month."),          # price change
    ("$79 per month.", "Custom pricing."),         # different topic
    ("30-day free trial.", "Free trial lasts 30 days."),   # pure rewording
    ("Track up to 10,000 events per month.", "Track unlimited events per month."),  # limit removed
]
vecs = model.encode([t for pair in probe_pairs for t in pair], normalize_embeddings=True)
for k, (a, b) in enumerate(probe_pairs):
    va, vb = vecs[2 * k], vecs[2 * k + 1]
    print(f"dot product {float(va @ vb):.3f}   {a!r} vs {b!r}")

dot product 0.958   '$79 per month.' vs '$89 per month.'
dot product 0.736   '$79 per month.' vs 'Custom pricing.'
dot product 0.921   '30-day free trial.' vs 'Free trial lasts 30 days.'
dot product 0.933   'Track up to 10,000 events per month.' vs 'Track unlimited events per month.'


Note the trap in the last row: the embedder scores the limit change at 0.93,
which is rewording territory. Embeddings alone would hide it. That is why the
script also runs two evidence checks on every matched pair.

In [5]:
from what_changed import extract_facts, extract_claims

a = "Refunds must be requested within 30 days."
b = "Refunds must be requested within 60 days."
va, vb = model.encode([a, b])
print(f"dot product {float(va @ vb):.3f}, almost a perfect rewording")
print("facts old:", dict(extract_facts(a)), "| facts new:", dict(extract_facts(b)))
print()
a = "SSO is not included in Growth."
b = "SSO is included in Growth."
va, vb = model.encode([a, b])
print(f"dot product {float(va @ vb):.3f}, and no number moved either")
print("claims old:", dict(extract_claims(a)), "| claims new:", dict(extract_claims(b)))

dot product 0.989, almost a perfect rewording
facts old: {'30': 1} | facts new: {'60': 1}

dot product 0.956, and no number moved either
claims old: {'negation': 1} | claims new: {}


## Step 3: label every clause

Matching rules, in order:
1. Identical text in both versions: **Unchanged**.
2. Closest old clause found by dot product, matched greedily, highest score wins the partner.
3. If the pair's numbers, prices, dates, percentages or durations differ: **Changed**, whatever the similarity.
4. If a risky claim word is added, removed or swapped (may contain, contains, free, guaranteed, up to, at least, unlimited, high in, source of, certified, negations): **Changed**, whatever the similarity. Negation words collapse into one class, so "no X" -> "without X" stays a rewording while "not included" -> "included" does not.
5. Similarity at or above the reword threshold (0.85): **Reworded**, hidden by default.
6. Related but below the threshold: **Changed**, but only when each clause is the other's overall nearest match across the whole other document; otherwise the two clauses stay unmatched.
7. No partner above the related floor (0.74): **Added** or **Removed**.

In [6]:
from what_changed import compare, render_report

pairs = compare(old_text, new_text, model, threshold=0.85, floor=0.74)
print(render_report(pairs, "examples/gtm_pricing_page/old.txt", "examples/gtm_pricing_page/new.txt",
                    "harims95/hobbylm-embed-140m", 0.85, 0.74, show_all=True))

WHAT REALLY CHANGED
old: examples/gtm_pricing_page/old.txt
new: examples/gtm_pricing_page/new.txt
model: harims95/hobbylm-embed-140m | reword threshold 0.85 | related floor 0.74

COUNTS
unchanged   15
reworded     5
changed      2
added        5
removed      0

CHANGED
1. [numbers/dates differ (old: $79 | new: $89), sim 0.96]
   old: $79 per month.
   new: $89 per month.
2. [numbers/dates differ (old: 10,000 | new: none), sim 0.93]
   old: Track up to 10,000 events per month.
   new: Track unlimited events per month.

ADDED
+ Scale plan
+ $199 per month.
+ Everything in Growth, with data residency in EU and US regions.
+ A named onboarding contact is included.
+ Nimbus is SOC 2 Type II certified.

REMOVED
(none)

REWORDED
1. [sim 0.98]
   old: Setup takes less than 15 minutes.
   new: Setup takes under 15 minutes.
2. [sim 0.92]
   old: Contact sales for a quote.
   new: Get in touch with our sales team for a quote.
3. [sim 0.92]
   old: 30-day free trial.
   new: Free trial lasts 30 da

## Step 4: the same test on all four example pairs

All sample data, all fictional companies:

In [7]:
import glob, os

for d in sorted(glob.glob("examples/*")):
    o = open(os.path.join(d, "old.txt")).read()
    n = open(os.path.join(d, "new.txt")).read()
    print("=" * 30, d)
    ps = compare(o, n, model, threshold=0.85, floor=0.74)
    print(render_report(ps, d + "/old.txt", d + "/new.txt",
                        "harims95/hobbylm-embed-140m", 0.85, 0.74, show_all=False))

============================== examples/ecommerce_supplier_spec


WHAT REALLY CHANGED
old: examples/ecommerce_supplier_spec/old.txt
new: examples/ecommerce_supplier_spec/new.txt
model: harims95/hobbylm-embed-140m | reword threshold 0.85 | related floor 0.74

COUNTS
unchanged    7
reworded     3  (hidden, run with --show-all to see them)
changed      5
added        0
removed      0

CHANGED
1. [numbers/dates differ (old: 26.5 | new: 27.5), sim 0.97]
   old: Dimensions: 26.5 cm tall, 7.2 cm diameter
   new: Dimensions: 27.5 cm tall, 7.2 cm diameter
2. [numbers/dates differ (old: 45 | new: 60), sim 0.97]
   old: Lead time: 45 days from confirmed purchase order
   new: Lead time: 60 days from confirmed purchase order
3. [numbers/dates differ (old: 500 | new: 250), sim 0.95]
   old: Minimum order quantity: 500 units
   new: Minimum order quantity: 250 units
4. [claim words differ (old: negation | new: none), sim 0.93]
   old: Care: hand wash only, do not use in a microwave
   new: Care instructions: hand wash only, keep away from microwave ovens
5. [claim

WHAT REALLY CHANGED
old: examples/gtm_pricing_page/old.txt
new: examples/gtm_pricing_page/new.txt
model: harims95/hobbylm-embed-140m | reword threshold 0.85 | related floor 0.74

COUNTS
unchanged   15
reworded     5  (hidden, run with --show-all to see them)
changed      2
added        5
removed      0

CHANGED
1. [numbers/dates differ (old: $79 | new: $89), sim 0.96]
   old: $79 per month.
   new: $89 per month.
2. [numbers/dates differ (old: 10,000 | new: none), sim 0.93]
   old: Track up to 10,000 events per month.
   new: Track unlimited events per month.

ADDED
+ Scale plan
+ $199 per month.
+ Everything in Growth, with data residency in EU and US regions.
+ A named onboarding contact is included.
+ Nimbus is SOC 2 Type II certified.

REMOVED
(none)

============================== examples/npd_product_brief


WHAT REALLY CHANGED
old: examples/npd_product_brief/old.txt
new: examples/npd_product_brief/new.txt
model: harims95/hobbylm-embed-140m | reword threshold 0.85 | related floor 0.74

COUNTS
unchanged   11
reworded     3  (hidden, run with --show-all to see them)
changed      1
added        1
removed      1

CHANGED
1. [numbers/dates differ (old: 190 | new: 170), sim 0.96]
   old: 190 calories per serving.
   new: 170 calories per serving.

ADDED
+ Now with chia seeds.

REMOVED
- Made in a facility that processes tree nuts.

============================== examples/sales_proposal_v7


WHAT REALLY CHANGED
old: examples/sales_proposal_v7/old.txt
new: examples/sales_proposal_v7/new.txt
model: harims95/hobbylm-embed-140m | reword threshold 0.85 | related floor 0.74

COUNTS
unchanged   11
reworded     4  (hidden, run with --show-all to see them)
changed      2
added        1
removed      1

CHANGED
1. [numbers/dates differ (old: 50%, 50% | new: 40%, 60%), sim 0.99]
   old: Payment terms: 50% upfront, 50% on final delivery.
   new: Payment terms: 40% upfront, 60% on final delivery.
2. [numbers/dates differ (old: 2 | new: none), sim 0.97]
   old: Deliverables: audit report, pilot model, rollout playbook, 2 training sessions.
   new: Deliverables: audit report, pilot model, rollout playbook.

ADDED
+ Also included: an executive dashboard for weekly forecast review, delivered with the pilot model.

REMOVED
- Phase 3: rollout support and training for the merchandising team, 4 weeks.



## Step 5: why the threshold sits at 0.85

Every matched pair across the four examples, sorted by similarity. Pure
rewordings cluster at the top; meaning changes are caught by the two evidence
checks whatever their score.

In [8]:
from what_changed import split_clauses, normalize

rows = []
for d in sorted(glob.glob("examples/*")):
    o = open(os.path.join(d, "old.txt")).read()
    n = open(os.path.join(d, "new.txt")).read()
    for p in compare(o, n, model, threshold=0.85, floor=0.74):
        if p.label in ("reworded", "changed"):
            rows.append((p.sim, p.label, p.reason))

rows.sort(key=lambda r: -r[0])
for sim, label, reason in rows:
    print(f"{sim:.3f}  {label:<9} {reason[:52]}")

0.988  changed   numbers/dates differ (old: 50%, 50% | new: 40%, 60%)
0.982  reworded  same meaning, different words
0.975  reworded  same meaning, different words
0.975  reworded  same meaning, different words
0.971  changed   numbers/dates differ (old: 26.5 | new: 27.5)
0.968  changed   numbers/dates differ (old: 45 | new: 60)
0.967  changed   numbers/dates differ (old: 2 | new: none)
0.965  reworded  same meaning, different words
0.961  reworded  same meaning, different words
0.960  changed   numbers/dates differ (old: 190 | new: 170)
0.959  reworded  same meaning, different words
0.958  changed   numbers/dates differ (old: $79 | new: $89)
0.947  changed   numbers/dates differ (old: 500 | new: 250)
0.944  reworded  same meaning, different words
0.942  reworded  same meaning, different words
0.933  changed   numbers/dates differ (old: 10,000 | new: none)
0.926  changed   claim words differ (old: negation | new: none)
0.925  reworded  same meaning, different words
0.921  reworded  sam

The pure rewordings land between 0.903 and 0.982. The meaning changes are
forced by the number and claim-words checks, so the threshold's remaining job is
to sort word-level noise from anything else. 0.85 was picked as the midpoint of
the gap measured before the claim-words check existed (lowest rewording 0.903,
highest non-numeric meaning change 0.783) and it still separates every pair in
these four documents correctly.

Sensitivity check: counts per label at other thresholds, all four example pairs.

In [9]:
from collections import Counter

for d in sorted(glob.glob("examples/*")):
    o = open(os.path.join(d, "old.txt")).read()
    n = open(os.path.join(d, "new.txt")).read()
    row = [os.path.basename(d).ljust(24)]
    for t in (0.70, 0.80, 0.85, 0.90, 0.95):
        c = Counter(p.label for p in compare(o, n, model, threshold=t, floor=0.74))
        row.append(f"{t:.2f}: ch{c['changed']} rw{c['reworded']}")
    print("  ".join(row))

ecommerce_supplier_spec   0.70: ch5 rw3  0.80: ch5 rw3  0.85: ch5 rw3  0.90: ch5 rw3  0.95: ch7 rw1


gtm_pricing_page          0.70: ch2 rw5  0.80: ch2 rw5  0.85: ch2 rw5  0.90: ch2 rw5  0.95: ch5 rw1


npd_product_brief         0.70: ch1 rw3  0.80: ch1 rw3  0.85: ch1 rw3  0.90: ch1 rw3  0.95: ch2 rw2


sales_proposal_v7         0.70: ch2 rw4  0.80: ch2 rw4  0.85: ch2 rw4  0.90: ch2 rw4  0.95: ch4 rw2


At 0.95 the harmless rewordings surface as Changed everywhere: the pricing
page report goes from 2 to 6 changed clauses and the signal drowns in word-level
noise. At 0.80 and below the counts stop changing on these documents, because
every meaning change here is now carried by the number and claim-words checks
rather than by a low threshold. On documents without such cues, a too-low
threshold would hide meaning flips, which is exactly what the hard cases below
measure.

## Step 6: hard cases, the pairs this tool exists for

`hard_cases.csv` files next to each example hold 33 pairs written to be
difficult on purpose: claim-strength changes with no number, negations, and
removed clauses. They were written once and never reworded after measuring.
Expected labels: changed or removed.

In [10]:
from what_changed import eval_hard_cases

total_hits, total_n = 0, 0
for d in sorted(glob.glob("examples/*")):
    hits, n = eval_hard_cases(os.path.join(d, "hard_cases.csv"), model, 0.85, 0.74,
                              claim_check=True, show_pairs=False)
    total_hits += hits
    total_n += n
print(f"hard cases with claim-words check: {total_hits}/{total_n} = {total_hits / total_n:.0%}")

HARD CASES: examples/ecommerce_supplier_spec/hard_cases.csv
accuracy: 7/8 = 88%



HARD CASES: examples/gtm_pricing_page/hard_cases.csv
accuracy: 8/9 = 89%



HARD CASES: examples/npd_product_brief/hard_cases.csv
accuracy: 8/8 = 100%



HARD CASES: examples/sales_proposal_v7/hard_cases.csv
accuracy: 7/8 = 88%

hard cases with claim-words check: 30/33 = 91%


In [11]:
total_hits, total_n = 0, 0
for d in sorted(glob.glob("examples/*")):
    hits, n = eval_hard_cases(os.path.join(d, "hard_cases.csv"), model, 0.85, 0.74,
                              claim_check=False, show_pairs=False)
    total_hits += hits
    total_n += n
print(f"hard cases without claim-words check: {total_hits}/{total_n} = {total_hits / total_n:.0%}")

HARD CASES: examples/ecommerce_supplier_spec/hard_cases.csv
accuracy: 3/8 = 38%



HARD CASES: examples/gtm_pricing_page/hard_cases.csv
accuracy: 4/9 = 44%



HARD CASES: examples/npd_product_brief/hard_cases.csv
accuracy: 4/8 = 50%



HARD CASES: examples/sales_proposal_v7/hard_cases.csv
accuracy: 3/8 = 38%

hard cases without claim-words check: 14/33 = 42%


The claim-words check takes hard-case accuracy from 42% to 91%. The 3
remaining misses are all the same shape: a condition quietly narrows ("every
plan" to "annual plans", "only" to "recommended", "included" to "for longer
engagements") with no number and no claim word from the list. The list is
configurable with --claim-words when a document type has its own risky phrases.

## Step 7: removed clauses vs the floor

A removed clause should stay Removed even when some leftover clause on the
other side is weakly related. Test: append each of the 10 removed clauses from
the hard cases to its role's old.txt and run the real pipeline. A raw max
similarity at or above the floor (0.74) is where the danger lives.

In [12]:
import csv

print(f"{'role':<26} {'max sim':>8}  {'in-doc label':<14} clause")
wrong = 0
total = 0
for d in sorted(glob.glob("examples/*")):
    old_doc = open(os.path.join(d, "old.txt")).read()
    new_doc = open(os.path.join(d, "new.txt")).read()
    doc_clauses = split_clauses(old_doc) + split_clauses(new_doc)
    doc_vecs = model.encode(doc_clauses)
    with open(os.path.join(d, "hard_cases.csv"), newline="") as f:
        for row in csv.DictReader(f):
            if row["expected"].strip() != "removed":
                continue
            clause = row["old"].strip()
            total += 1
            v = model.encode([clause])[0]
            top = float((doc_vecs @ v).max())
            ps = compare(old_doc + "\n" + clause + "\n", new_doc, model, 0.85, 0.74)
            mine = [p for p in ps if p.old and normalize(p.old).lower() == normalize(clause).lower()]
            label = mine[0].label if mine else "?"
            if label != "removed":
                wrong += 1
            print(f"{os.path.basename(d):<26} {top:>8.3f}  {label:<14} {clause[:44]!r}")
print(f"\nresult: {total - wrong}/{total} shown as Removed, {wrong} wrongly matched")

role                        max sim  in-doc label   clause


ecommerce_supplier_spec       0.792  removed        'Sample units available on request.'


ecommerce_supplier_spec       0.711  removed        'Spares kit available for order.'


gtm_pricing_page              0.747  removed        'Live chat support is available in English.'


gtm_pricing_page              0.744  removed        'Prices exclude local taxes.'


gtm_pricing_page              0.718  removed        'We never sell customer data.'


npd_product_brief             1.000  removed        'Made in a facility that processes tree nuts.'


npd_product_brief             0.720  removed        'Refrigerate after opening.'


sales_proposal_v7             0.828  removed        'The pilot covers 2 product categories.'


sales_proposal_v7             0.727  removed        'References from 3 retail clients available o'


sales_proposal_v7             0.769  removed        'A dedicated Slack channel is included.'

result: 10/10 shown as Removed, 0 wrongly matched


Five of the ten removed clauses have a raw max similarity at or above the
floor. Before the mutual-nearest rule, one of them ("A dedicated Slack channel is
included.") grabbed the unrelated leftover "Also included: an executive
dashboard..." at 0.769 and showed up as a fake Changed pair. The fix: below the
reword threshold, a pair is only allowed when each clause is the other's overall
nearest match across the whole other document. With that rule, all 10 stay Removed,
and the true low-similarity pairs in the clean examples (the lid material swap
at 0.783) still pair, because there each side's overall nearest match really is the
other. The floor itself stays at 0.74: raising it would push the lid pair into
Added, which is worse.

## Limits, shown honestly

**The number and claim checks only see what they list.** A condition that
narrows with no number and no claim word still hides: "Free migration is
included on every plan." to "...on annual plans." scores 0.96 and is reported as
Reworded by this pipeline. Same for "Hand wash only." to "Hand wash
recommended." and "Free onboarding workshop included." to "...for longer
engagements." For high-stakes documents, run --show-all and skim the Reworded
list yourself. This is a screening pass, not a sign-off.

In [13]:
va, vb = model.encode(["Free migration is included on every plan.",
                       "Free migration is included on annual plans."])
print(f"dot product {float(va @ vb):.3f}, hidden as a rewording by this pipeline")

dot product 0.957, hidden as a rewording by this pipeline


Other limits:
- Claim words are a fixed list, tuned for offers, specs, proposals and label
  copy. Your domain may need different risky phrases (--claim-words).
- The number check reads "May" as the month, so "may contain" also trips it.
  That accident is harmless here: a permission word appearing or vanishing is
  usually a real change.
- Clause splitting is line and sentence based. Bullet-style documents split
  cleanly; a wall of prose may split at odd places or stay too long per clause.
- The model reads 512 tokens per clause by default, up to 2,048 if you raise
  the limit. Longer clauses are truncated
  and the script prints a warning to stderr.
- English only. The recipe and thresholds were checked on English text.
- Greedy matching is order dependent. When two clauses both want the same
  partner, the higher score wins and the other side can fall through to
  Added/Removed.

## Run it yourself

```
python what_changed.py examples/gtm_pricing_page/old.txt examples/gtm_pricing_page/new.txt
python what_changed.py old.txt new.txt --show-all
python what_changed.py old.txt new.txt --threshold 0.80
python what_changed.py --hard-cases examples/npd_product_brief/hard_cases.csv
python what_changed.py old.txt new.txt --claim-words "free,guaranteed,not"
python what_changed.py old.txt new.txt --no-claim-check
```